## Импорты и загрузка конфига

In [ ]:
%pip install pandas torch transformers datasets evaluate sacrebleu bert-score sentence-transformers rapidfuzz deep-translator tqdm numpy matplotlib PyYAML

In [ ]:
try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

In [ ]:
if is_colab:
    !git clone https://github.com/AbonentVneSeti/text_processing_final_project
    %cd text_processing_final_project

In [ ]:
import sys
sys.path.append('..')
from utils import load_config, build_dataloaders
import pandas as pd
import importlib

import warnings
warnings.filterwarnings("ignore")

config = load_config("config.yaml")
print("Config loaded:")
print(config)

## Получение и обработка датасета

In [ ]:
source = config["dataset_source"]
source_params = config["source_params"].get(source, {})
module = importlib.import_module(f"dataset.create_dataset.{source}")
df = getattr(module, "load_or_create")(source_params)
df = df.sample(n=len(df)//2, random_state=42)
#df = df.sample(n=1000, random_state=42)

print(f"Dataset size: {len(df)}")
df.head()

In [ ]:
from dataset.prepare_dataset.prepare import prepare_dataset
df_clean = prepare_dataset(df, config["preprocessing"])
print(f"Clean dataset size: {len(df_clean)}")

In [ ]:
train_loader, val_loader = build_dataloaders(df_clean, config["model_config"])
print(f"Train samples: {len(train_loader.dataset)}, Val samples: {len(val_loader.dataset)}")

## Создание модели и обучение

In [ ]:
from models.rut5_paraphraser.model import ParaphraserModel
from models.trainer import train_model

model = ParaphraserModel(config["model_config"], device="cuda" if __import__('torch').cuda.is_available() else "cpu")
trained_model = train_model(
    model,
    train_loader,
    val_loader,
    config["model_config"],
    config["trainer_config"],
    config["metrics_config"]
)
trained_model.save(config["trainer_config"]["output_dir"])

In [ ]:
if is_colab:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

    !cp -r "models/rut5_paraphraser/saves" "/content/drive/MyDrive/colab_export/"

    from google.colab import runtime
    runtime.unassign()


## Оценка доп. метриками

In [ ]:
 
from models.model_use import load_model, evaluate_model

model = load_model(
    config["model_name"],
    config["model_config"],
    checkpoint_path=config["trainer_config"]["output_dir"]
)

test_metrics = evaluate_model(model, val_loader, config["metrics_config"])
print("Test metrics:", test_metrics)